# Explore the memory usage of xarray and dask

 - Author: Lucas Bastien.
 - Copyright (c) 2025-now, Institut des Géosciences de l'Environnement, (IGE, UMR 5001), Grenoble, France.
 - License: [BSD 3-clause "New" or "Revised" license](https://spdx.org/licenses/BSD-3-Clause.html).
 
 We analyze the memory usage of xarray and dask using an arbitrary output file from the WRF-Chem model.
 

# Dependencies

Dependencies for this notebook are fairly straight-forward except for the last section that uses the wrfpp module that is available [here](https://github.com/Regional-Modeling-LATMOS-IGE/WRF-infra/blob/main/postprocess/wrfpp.py).

In [1]:
# Modules from the Python standard library
import sys
import tracemalloc
import functools

# 3rd-party dependencies (import dask explicitly to make sure it is available)
import dask
import numpy as np
import xarray as xr

# The following is used only in the last section
import wrfpp

# Print system and package information
import watermark
%load_ext watermark
%watermark --machine --python --iversions

Python implementation: CPython
Python version       : 3.13.2
IPython version      : 9.0.2

Compiler    : Clang 18.1.8 
OS          : Darwin
Release     : 25.0.0
Machine     : arm64
Processor   : arm
CPU cores   : 8
Architecture: 64bit

xarray   : 2025.3.1
dask     : 2025.9.1
sys      : 3.13.2 | packaged by conda-forge | (main, Feb 17 2025, 14:02:48) [Clang 18.1.8 ]
watermark: 2.5.0
numpy    : 2.2.4



# Define monitoring tools

It is important to note that the function `sys.getsizeof` returns the self-declared size of the object (this function calls the `__sizeof__` method of the object). Acccording to [the Python documentation](https://docs.python.org/3/library/sys.html), _"All built-in objects will return correct results, but this does not have to hold true for third-party extensions"_ (it is up to the developpers to implement `__sizeof__` correctly).

However, our `peak_memory_usage` decorator should be true and unbiased because it monitors the actual memory usage of the running Python process.

In [2]:
def print_function_name(func):
    """Decorator to print the function's name when it is called."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("* %s *" % func.__name__)
        func(*args, **kwargs)

    return wrapper


def sizeof(object):
    """Print the self-declared size of given object."""
    print(
        "Self-declared memory usage of object: %.0f Kbytes"
        % (sys.getsizeof(object) / 1000)
    )


def peak_memory_usage(func):
    """Decorator to print the peak memory usage of a function call."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        tracemalloc.start()
        func(*args, **kwargs)
        stats = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        print("Peak memory usage: %.0f Kbytes\n" % (stats[1] / 1000))

    return wrapper

# Memory usage when manipulating variables

We are mostly interested in the memory usage associated with:
 1. Accessing data from the data set
 2. Calculating derived data from what can be found in the data set

In [3]:
def native_variable_summary(ds, varname):
    sizeof(ds[varname])


def native_variable_sliced(ds, varname):
    sizeof(ds[varname][:])


def native_variable_values(ds, varname):
    sizeof(ds[varname].values)


def native_variable_all_values_sliced(ds, varname):
    sizeof(ds[varname].values[:])


def native_variable_copy_all_values(ds, varname):
    sizeof(np.array(ds[varname].values))


def derived_variable_summary(ds, varname):
    sizeof(ds[varname] + 1)


def derived_variable_all_values(ds, varname):
    sizeof(ds[varname].values + 1)


def derived_variable_copy_all_values(ds, varname):
    sizeof(np.array(ds[varname].values + 1))


all_test_functions = [
    native_variable_summary,
    native_variable_sliced,
    native_variable_values,
    native_variable_all_values_sliced,
    native_variable_copy_all_values,
    derived_variable_summary,
    derived_variable_all_values,
    derived_variable_copy_all_values,
]

# Decorate all these functions with our analytics tools
for i, func in enumerate(all_test_functions):
    all_test_functions[i] = print_function_name(peak_memory_usage(func))

## Without dask

From the piece of code below we conclude the following:

 - Accessing a variable via `ds[varname]` does not load the values into memory. This is expected since `ds[varname]` should just be a reference to the corresponding `DataArray`. Slicing the `DataArray` with `[:]` but without using `.values` does not trigger the loading of the data into memory either.

 - Accessing the data (via `.values`, or `.values[:]`) does evidently load the data. In fact, peak memory usage is roughly twice the size of the entire array.

 - Defining a derived variable (eg `ds[varname].values + 1`) does trigger the loading of the entire array.

In [4]:
varname = "QVAPOR"

for func in all_test_functions:
    ds = xr.open_dataset("wrfout.nc", chunks=None)
    func(ds, varname)
    ds.close()

* native_variable_summary *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 1 Kbytes

* native_variable_sliced *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 9 Kbytes

* native_variable_values *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3848 Kbytes

* native_variable_all_values_sliced *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3847 Kbytes

* native_variable_copy_all_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 3847 Kbytes

* derived_variable_summary *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3847 Kbytes

* derived_variable_all_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 3847 Kbytes

* derived_variable_copy_all_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 5764 Kbytes



## With dask

From the tests below, we see that a major difference when using dask is that defining a derived variable without accessing its values with `.values` does not load the entire array into memory. According to the [xarray official tutorial](https://tutorial.xarray.dev/intermediate/xarray_and_dask.html), _"Xarray seamlessly wraps dask so all computation is deferred until explicitly requested"_.



In [5]:
varname = "QVAPOR"

for func in all_test_functions:
    ds = xr.open_dataset("wrfout.nc", chunks="auto")
    func(ds, varname)
    ds.close()

* native_variable_summary *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 1 Kbytes

* native_variable_sliced *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 10 Kbytes

* native_variable_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 12927 Kbytes

* native_variable_all_values_sliced *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3885 Kbytes

* native_variable_copy_all_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 3884 Kbytes

* derived_variable_summary *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 141 Kbytes

* derived_variable_all_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 3884 Kbytes

* derived_variable_copy_all_values *
Self-declared memory usage of object: 1921 Kbytes
Peak memory usage: 3884 Kbytes



# Caching and referencing 

Whether with or without using dask, some caching and referencing of already-loaded data is happening: the data is not loaded into memory from scratch everytime it is called upon (cf. last example below). We do not discuss this further here.

In [6]:
@peak_memory_usage
def native_variable_all_values_sliced(ds, varname):
    sizeof(ds[varname].values[:])


varname = "QVAPOR"

print("Twice in a row, closing the file in between:\n")
for i in range(2):
    ds = xr.open_dataset("wrfout.nc")
    native_variable_all_values_sliced(ds, varname)
    ds.close()

print("\nTwice in a row, without closing the file in between:\n")
ds = xr.open_dataset("wrfout.nc")
for i in range(2):
    native_variable_all_values_sliced(ds, varname)
ds.close()

Twice in a row, closing the file in between:

Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3846 Kbytes

Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3846 Kbytes


Twice in a row, without closing the file in between:

Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 3846 Kbytes

Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 4 Kbytes



# String representation of WRF derived variables with wrfpp

The idea here is to check whether our implementation of the string representation of WRF derived variables in wrfpp triggers the loading of the data into memory.

The answer is quite short (see tests below for details): the data is loaded into memory when not using dask, and is not loaded into memory when using dask.

In [7]:
def variable_summary(ds, varname):
    sizeof(getattr(ds, varname))


def variable_str_representation(ds, varname):
    sizeof(getattr(ds, varname).__str__())


all_test_functions = [
    variable_summary,
    variable_str_representation,
]

# Decorate all these functions with our analytics tools
for i, func in enumerate(all_test_functions):
    all_test_functions[i] = print_function_name(peak_memory_usage(func))

## Without dask

In [8]:
varname = "air_temperature"

for func in all_test_functions:
    ds = xr.open_dataset("wrfout.nc", chunks=None).wrf
    func(ds, varname)
    ds.close()

* variable_summary *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 2 Kbytes

* variable_str_representation *
Self-declared memory usage of object: 2 Kbytes
Peak memory usage: 7919 Kbytes



## With dask

In [9]:
varname = "air_temperature"

for func in all_test_functions:
    ds = xr.open_dataset("wrfout.nc", chunks="auto").wrf
    func(ds, varname)
    ds.close()

* variable_summary *
Self-declared memory usage of object: 0 Kbytes
Peak memory usage: 1 Kbytes

* variable_str_representation *
Self-declared memory usage of object: 1 Kbytes
Peak memory usage: 590 Kbytes

